In [0]:
# 1. Configuración de Rutas (Laboratorio Aislado)
base_lab = "/Volumes/ecomotive/landing/ecomotive_vol/lab_rescue_mode"
path_inbox = f"{base_lab}/inbox"
path_chk   = f"{base_lab}/checkpoints"
path_sch   = f"{base_lab}/schemas"
target_table = "ecomotive.landing.lab_rescue_test"

# Limpieza inicial
spark.sql(f"DROP TABLE IF EXISTS {target_table}")
dbutils.fs.rm(base_lab, True)
dbutils.fs.mkdirs(path_inbox)

# 2. Creamos un archivo "BASE" (Batch 01) solo para fijar el esquema correcto
# Este archivo es "santo", tiene lo que queremos.
json_base = '{"id": 1, "velocidad": 100, "rpm": 2000}'
dbutils.fs.put(f"{path_inbox}/base.json", json_base, overwrite=True)

print("Laboratorio listo. Esquema base esperando.")

In [0]:
# LEEMOS CON EL ESCUDO ACTIVADO
df_rescue = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.inferColumnTypes", "true")
  
  # --- LA CONFIGURACIÓN ESTRELLA ---
  # Le decimos: "Si ves columnas nuevas, NO cambies la tabla. Guárdalas en el rescate."
  .option("cloudFiles.schemaEvolutionMode", "rescue") 
  .option("cloudFiles.rescuedDataColumn", "_rescued_data")
  .option("cloudFiles.schemaLocation", path_sch)
  
  .load(path_inbox)
)

# Escribimos
query = (df_rescue.writeStream
  .format("delta")
  .option("checkpointLocation", path_chk)
  # Nota: Aunque pongas mergeSchema, el modo 'rescue' del lector manda sobre qué columnas pasan
  .option("mergeSchema", "true") 
  .trigger(availableNow=True)
  .table(target_table)
)

query.awaitTermination()
print("Proceso terminado con Modo Rescue.")

In [0]:
# --- BATCH "INTRUSO" ---
# Fila 1: Tiene una columna EXTRA 'presion_ruedas' (Schema Drift)
# Fila 2: Tiene un ERROR de tipo en 'rpm' ("MIL")
json_intruso = """
{"id": 2, "velocidad": 80, "rpm": 1500, "presion_ruedas": 3.5}
{"id": 3, "velocidad": 90, "rpm": "MIL", "presion_ruedas": 3.0}
"""

dbutils.fs.put(f"{path_inbox}/intruso.json", json_intruso.strip(), overwrite=True)

print("Batch Intruso generado: Trae columnas nuevas y errores.")

In [0]:
%sql
SELECT * FROM ecomotive.landing.lab_rescue_test ORDER BY id;

In [0]:
%sql
DESCRIBE HISTORY ecomotive.landing.lab_rescue_test;